# Template 00: EDA for All Features

**Purpose:** Analyze all columns and generate encoding recommendations

**Outputs:**
- `config_generated/auto_feature_encoding.csv` - Auto-detected encoding recommendations
- `config_generated/auto_dtype_fixes.csv` - Auto-detected dtype conversions
- `config_generated/master_feature_encoding.csv` - Merged manual + auto (used by pipeline)
- `config_generated/master_dtype_fixes.csv` - Merged manual + auto (used by pipeline)
- Visualizations (histograms and bar charts)

**Note:** Run this manually (not part of automated pipeline)

In [ ]:
# Parameters (can be overridden)
config_path = "config/car_coll/v1"
sample_size = 30000  # Number of rows to sample for analysis

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import yaml
import os
import sys
from pathlib import Path

# Find project root (works from templates/ or output/model/v1/notebooks/)
current_dir = Path.cwd()
if current_dir.name == 'templates':
    project_root = current_dir.parent
elif current_dir.name == 'notebooks':
    # From output/model/v1/notebooks -> go up 4 levels to project root
    project_root = current_dir.parent.parent.parent.parent
else:
    project_root = current_dir

# Add lib to path
lib_path = str(project_root / 'lib')
if lib_path not in sys.path:
    sys.path.insert(0, lib_path)

from utils import setup_notebook_environment, load_config

print("="*70)
print("EDA FOR ALL FEATURES")
print("="*70)
print(f"Project root: {project_root}")
print(f"Config path: {config_path}")
print(f"Sample size: {sample_size:,} rows")

In [ ]:
# Load config
config_file = f"{config_path}/config.yaml"
cfg = load_config(config_file)

# Setup paths
config_gen_dir = f"{config_path}/config_generated"
os.makedirs(config_gen_dir, exist_ok=True)

print(f"\n✓ Config loaded: {cfg['experiment']['name']}")
print(f"✓ Generated config dir: {config_gen_dir}")

## Step 1: Load Column Metadata

In [ ]:
# Load all_columns_master.csv
master_cols_file = f"{config_path}/all_columns_master.csv"
all_cols_df = pd.read_csv(master_cols_file)

print(f"\n✓ Loaded {len(all_cols_df)} columns from all_columns_master.csv")
print(f"\nColumn types:")
print(all_cols_df['dtype'].value_counts())

all_cols_df.head()

## Step 2: Load Sample Data

In [ ]:
# Load sample data
data_root = cfg['paths']['master_data_path']
master_file = f"{data_root}/{cfg['data']['master_file']}"

print(f"\n📂 Loading sample data from: {master_file}")
print(f"   Sample size: {sample_size:,} rows")

# Read parquet with row limit
df_sample = pd.read_parquet(master_file).head(sample_size)

print(f"\n✓ Loaded sample: {df_sample.shape}")
print(f"  Memory: {df_sample.memory_usage(deep=True).sum() / 1e6:.1f} MB")

## Step 3: Analyze Columns

In [ ]:
from eda_helpers import analyze_column, detect_dtype_conversions, merge_encodings

results = []
target_exposure = {cfg['experiment']['target'], cfg['experiment']['exposure']}

for _, row in all_cols_df.iterrows():
    col = row['column_name']
    if col in df_sample.columns:
        results.append(analyze_column(col, df_sample[col], target_exposure=target_exposure))

auto_df = pd.DataFrame(results)
print(f'Analyzed {len(auto_df)} columns')

## Step 4: Save & Merge

In [ ]:
auto_df.to_csv(f'{config_gen_dir}/auto_feature_encoding.csv', index=False)
print(f'Saved auto_feature_encoding.csv')

dtype_df = detect_dtype_conversions(df_sample)
if len(dtype_df) > 0:
    dtype_df.to_csv(f'{config_gen_dir}/auto_dtype_fixes.csv', index=False)

manual_file = f'{config_path}/manual_feature_encoding.csv'
master_df = merge_encodings(auto_df, manual_file)
master_df.to_csv(f'{config_gen_dir}/master_feature_encoding.csv', index=False)
print(f'Saved master_feature_encoding.csv')

master_df.head(20)

## Summary

In [ ]:
print(f'Total: {len(master_df)}')
print(f"Manual: {(master_df['source']=='manual').sum()}")
print(f"Auto: {(master_df['source']=='auto').sum()}")
print(f"\nEncoding types:")
print(master_df['encoding'].value_counts())